# Evaluate Object-Agnostic Prompts (frozen WinCLIP vs learned)

Scores prompt checkpoints on the **evaluation half** of the protocol split -- the
images the attack pipeline holds out and the prompts were never fitted on. No
training; one visual forward pass per image, reused for every prompt set.

It answers two questions:

**(a) Does training help at all?** Learned prompts against the frozen WinCLIP
ensemble they replace, scored through exactly the same path.

**(b) Does the run-to-run noise matter?** Any two learned checkpoints from the
same dataset and protocol (for example an old and a new `balanced/mvtec`) are
compared directly. They were fitted on identical images, so any difference
between them is seed/nondeterminism variance.

| result | meaning | action |
| --- | --- | --- |
| learned > frozen, repeats agree | it works, noise is cosmetic | ship it |
| learned > frozen, repeats differ | real but noisy | fix determinism, report variance |
| learned <= frozen | prompts are not earning their place | fix training |

Before running: enable a **GPU**, attach MVTec AD and VisA, attach your prompt
checkpoints as a Kaggle dataset, and enable Internet. Edit only the settings cell.

**Each protocol is scored on its own evaluation half.** `balanced` and `full`
hold out different images, and the `balanced` training half overlaps the `full`
evaluation half -- so scoring a `balanced` checkpoint on the `full` half would
leak. Comparisons are therefore only made within a (dataset, protocol) group.

In [ ]:
# ========================= USER SETTINGS =========================
PROJECT_GIT_URL = "https://github.com/Parsagh05/object-agnostic-prompt-training.git"  # This prompt-training pipeline.
PROJECT_BRANCH = "main"
PROJECT_SOURCE = None  # None clones GitHub; otherwise an attached folder/ZIP under /kaggle/input.

ANOMALYCLIP_GIT_URL = "https://github.com/zqhang/AnomalyCLIP.git"  # Official source used to load public CLIP.
ANOMALYCLIP_COMMIT = "3911738c0867544f545a076ad78f3f11d9ecbfdf"
ANOMALYCLIP_SOURCE = None

# The frozen ensemble is imported from the attack pipeline itself, so the
# baseline is exactly the vocabulary the surrogate uses -- never a copy.
ATTACK_GIT_URL = "https://github.com/Parsagh05/adversarial-perturbation-generation.git"
ATTACK_BRANCH = "main"
ATTACK_SOURCE = None

MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"

# None auto-discovers every prompt checkpoint under /kaggle/input and names each
# by its last two folders, e.g. "balanced/mvtec" or "prompts_old/mvtec".
# Otherwise give an explicit {name: path} mapping.
PROMPT_CHECKPOINTS = None

CLIP_WEIGHTS = None  # None downloads ViT-L/14@336px; set an attached .pt path when Internet is off.

SPLIT_SEED = 111  # Must match the training runs and the attack pipeline.
EVALUATION_FRACTION = 0.5  # Must match both as well.

MAP_RES = 256  # Resolution pixel metrics are computed at. Applied identically to every prompt set.
GAUSSIAN_SIGMA = 4.0  # AnomalyCLIP's inference smoothing, specified at 518px and rescaled to MAP_RES. 0 disables.
BATCH_SIZE = 8  # No backward pass here, so this can exceed the training batch size.
LIMIT_PER_CATEGORY = None  # Set e.g. 4 for a fast smoke test of the whole notebook.

OUTPUT_DIR = "/kaggle/working/prompt_evaluation"
# ================================================================

In [ ]:
# Prepare repositories and Python dependencies.
from pathlib import Path
import os, shutil, subprocess, sys, zipfile
import torch

assert Path("/kaggle/working").is_dir(), "This notebook is intended for Kaggle."
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before loading ViT-L/14@336px.")
print("GPU:", torch.cuda.get_device_name(0))

WORK_ROOT = Path("/kaggle/working/prompt_evaluation_work")
PROJECT_ROOT = WORK_ROOT / "project"
ANOMALYCLIP_ROOT = WORK_ROOT / "AnomalyCLIP"
ATTACK_ROOT = WORK_ROOT / "attack"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def materialize_source(source, destination, git_url, branch_or_commit, *, is_commit=False):
    destination = Path(destination)
    if destination.exists():
        print("Reusing:", destination)
    elif source:
        source_path = Path(source)
        if source_path.is_dir():
            shutil.copytree(source_path, destination)
        elif source_path.is_file() and source_path.suffix.lower() == ".zip":
            destination.mkdir(parents=True)
            with zipfile.ZipFile(source_path) as archive:
                archive.extractall(destination)
            children = [p for p in destination.iterdir() if p.is_dir()]
            if len(children) == 1 and not (destination / "pyproject.toml").exists():
                nested = children[0]
                for item in nested.iterdir():
                    shutil.move(str(item), destination / item.name)
                nested.rmdir()
        else:
            raise FileNotFoundError(f"Invalid attached source: {source_path}")
    else:
        command = ["git", "clone"]
        if not is_commit:
            command += ["--branch", branch_or_commit, "--depth", "1"]
        command += [git_url, str(destination)]
        subprocess.check_call(command)
    if is_commit and (destination / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(destination), "checkout", branch_or_commit])
    return destination

materialize_source(PROJECT_SOURCE, PROJECT_ROOT, PROJECT_GIT_URL, PROJECT_BRANCH)
materialize_source(ANOMALYCLIP_SOURCE, ANOMALYCLIP_ROOT, ANOMALYCLIP_GIT_URL,
                   ANOMALYCLIP_COMMIT, is_commit=True)
materialize_source(ATTACK_SOURCE, ATTACK_ROOT, ATTACK_GIT_URL, ATTACK_BRANCH)

assert (PROJECT_ROOT / "pyproject.toml").is_file(), f"Project checkout incomplete: {PROJECT_ROOT}"
assert (ANOMALYCLIP_ROOT / "AnomalyCLIP_lib").is_dir(), f"AnomalyCLIP checkout incomplete: {ANOMALYCLIP_ROOT}"
assert (ATTACK_ROOT / "adversarial_harness" / "prompts.py").is_file(), f"Attack checkout incomplete: {ATTACK_ROOT}"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "thop>=0.1.1"])
import thop  # AnomalyCLIP imports it at module import time.

# Editable installs are invisible to an already-running kernel, so add src directly.
for extra in (str(PROJECT_ROOT / "src"), str(ATTACK_ROOT)):
    if extra not in sys.path:
        sys.path.insert(0, extra)
import importlib
importlib.invalidate_caches()

from object_agnostic_prompt_attack.data import (
    PromptTrainingDataset, automatic_protocol_split, discover_dataset,
)
from object_agnostic_prompt_attack.checkpoint import (
    load_prompt_checkpoint, restore_prompt_checkpoint,
)
from object_agnostic_prompt_attack.config import ModelConfig, PromptConfig
from object_agnostic_prompt_attack.model import build_public_clip_prompt_model

# The exact frozen vocabulary the surrogate uses, imported rather than copied.
from adversarial_harness.prompts import (
    ABNORMAL_STATES, NORMAL_STATES, PREFIX_TEMPLATES,
    cartesian_prompts, frozen_ensemble_sha256,
)
print(f"\nfrozen ensemble: {len(PREFIX_TEMPLATES)} templates x "
      f"({len(NORMAL_STATES)} normal, {len(ABNORMAL_STATES)} abnormal) = "
      f"{len(PREFIX_TEMPLATES)*len(NORMAL_STATES)} + {len(PREFIX_TEMPLATES)*len(ABNORMAL_STATES)} prompts")
print("ensemble sha256:", frozen_ensemble_sha256()[:16], "(pinned by the attack repo)")
print("ready.")

In [ ]:
# Locate datasets and the prompt checkpoints to score.
import json

INPUT_ROOT = Path("/kaggle/input")

def find_mvtec_root():
    matches = sorted({p.parents[2] for p in INPUT_ROOT.rglob("bottle/test/good") if p.is_dir()})
    if len(matches) != 1:
        raise RuntimeError(f"Set MVTEC_ROOT; candidates: {matches}")
    return matches[0]

def find_visa_root():
    matches = sorted({p.parents[1] for p in INPUT_ROOT.rglob("split_csv/1cls.csv") if p.is_file()})
    if len(matches) != 1:
        raise RuntimeError(f"Set VISA_ROOT; candidates: {matches}")
    return matches[0]

ROOTS = {
    "mvtec": Path(MVTEC_ROOT) if MVTEC_ROOT else find_mvtec_root(),
    "visa": Path(VISA_ROOT) if VISA_ROOT else find_visa_root(),
}
print("MVTec root:", ROOTS["mvtec"])
print("VisA  root:", ROOTS["visa"])

def discover_checkpoints():
    """Every valid prompt checkpoint under /kaggle/input, named by its folders."""
    found = {}
    for path in sorted(INPUT_ROOT.rglob("*.pt")):
        try:
            payload = load_prompt_checkpoint(path)
        except Exception:
            continue
        name = "/".join(path.parts[-3:-1])
        found[name] = (path, payload)
    return found

if PROMPT_CHECKPOINTS:
    CHECKPOINTS = {n: (Path(p), load_prompt_checkpoint(p)) for n, p in PROMPT_CHECKPOINTS.items()}
else:
    CHECKPOINTS = discover_checkpoints()

if not CHECKPOINTS:
    raise RuntimeError(
        "No prompt checkpoints found under /kaggle/input. Attach the folder "
        "containing prompts_epoch15.pt, or set PROMPT_CHECKPOINTS explicitly."
    )

# Group by (dataset, protocol): each group is scored on its own evaluation half.
GROUPS = {}
print(f"\n{'name':26s} {'dataset':8s} {'protocol':10s} {'n_ctx':>5s}  cohort sha")
for name, (path, payload) in sorted(CHECKPOINTS.items()):
    config = payload["prompt_config"]
    dataset = str(payload["dataset"]).lower()
    # Checkpoints written before split_protocol existed are all balanced.
    protocol = str(config.get("split_protocol", "balanced"))
    if config.get("n_ctx") != 12 or config.get("normal_suffix") != "object.":
        raise RuntimeError(f"{name}: unexpected prompt architecture {dict(config)}")
    GROUPS.setdefault((dataset, protocol), []).append(name)
    print(f"{name:26s} {dataset:8s} {protocol:10s} {config['n_ctx']:5d}  "
          f"{payload['sample_manifest_sha256'][:12]}")

print(f"\n{len(CHECKPOINTS)} checkpoint(s) in {len(GROUPS)} group(s): "
      + ", ".join(f"{d}/{p} ({len(v)})" for (d, p), v in sorted(GROUPS.items())))

In [ ]:
# Build the evaluation half for each (dataset, protocol) group.
EVAL_SETS = {}
print(f"{'group':18s} {'train':>7s} {'eval':>7s} {'eval norm/abn':>15s} {'categories':>11s}")
for (dataset, protocol) in sorted(GROUPS):
    samples = discover_dataset(dataset, ROOTS[dataset])
    train, evaluation = automatic_protocol_split(
        samples, seed=SPLIT_SEED, evaluation_fraction=EVALUATION_FRACTION,
        protocol=protocol,
    )
    if LIMIT_PER_CATEGORY:
        capped, seen = [], {}
        for s in evaluation:
            key = (s.category, s.label)
            if seen.get(key, 0) < LIMIT_PER_CATEGORY:
                seen[key] = seen.get(key, 0) + 1
                capped.append(s)
        evaluation = capped
    EVAL_SETS[(dataset, protocol)] = evaluation
    n_normal = sum(s.label == 0 for s in evaluation)
    print(f"{dataset+'/'+protocol:18s} {len(train):7d} {len(evaluation):7d} "
          f"{n_normal:7d}/{len(evaluation)-n_normal:<7d} "
          f"{len({s.category for s in evaluation}):11d}")

# The checkpoints must never have seen these images.
for (dataset, protocol), evaluation in EVAL_SETS.items():
    train, _ = automatic_protocol_split(
        discover_dataset(dataset, ROOTS[dataset]), seed=SPLIT_SEED,
        evaluation_fraction=EVALUATION_FRACTION, protocol=protocol)
    leak = {s.protocol_id for s in train} & {s.protocol_id for s in evaluation}
    if leak:
        raise RuntimeError(f"{dataset}/{protocol}: {len(leak)} evaluation images were trained on")
print("\nleakage check passed: no evaluation image appears in any training cohort")

In [ ]:
# Load the frozen surrogate: public CLIP, no DPAM -- the same path training used.
model = build_public_clip_prompt_model(
    PromptConfig(),
    ModelConfig(
        anomalyclip_root=str(ANOMALYCLIP_ROOT),
        clip_download_root=str(WORK_ROOT / "clip_cache"),
        clip_model_name=str(CLIP_WEIGHTS) if CLIP_WEIGHTS else "ViT-L/14@336px",
        device="cuda",
    ),
)
model.assert_parameter_boundary()
DEVICE = model.device
IMAGE_SIZE = model.model_config.image_size
print("surrogate ready:", IMAGE_SIZE, "px |", len(model.model_config.feature_layers),
      "feature layers", model.model_config.feature_layers, "| DPAM:", model.model_config.use_dpam)

# AnomalyCLIP's tokenizer, from the checkout the model was loaded through.
import prompt_ensemble as anomalyclip_prompts
tokenize = anomalyclip_prompts.tokenize

import torch.nn.functional as F

@torch.no_grad()
def encode_prompt_texts(prompts):
    """One normalized mean prototype for a list of prompt strings."""
    embeddings = []
    for start in range(0, len(prompts), 64):
        tokens = tokenize(list(prompts[start:start + 64])).to(DEVICE)
        embeddings.append(F.normalize(model.clip_model.encode_text(tokens).float(), dim=-1))
    stacked = torch.cat(embeddings, dim=0)
    return F.normalize(stacked.mean(dim=0), dim=-1)

@torch.no_grad()
def frozen_text_for(category):
    """The attack pipeline's two WinCLIP prototypes for one category."""
    normal = encode_prompt_texts(cartesian_prompts(category, NORMAL_STATES))
    abnormal = encode_prompt_texts(cartesian_prompts(category, ABNORMAL_STATES))
    return torch.stack([normal, abnormal], dim=0)

@torch.no_grad()
def learned_text_for(path):
    """Encode a saved context pair through the same frozen text encoder."""
    restore_prompt_checkpoint(model.prompt_learner, path)
    return model.encode_prompts().detach()

LEARNED_TEXT = {name: learned_text_for(path) for name, (path, _) in CHECKPOINTS.items()}
for name, text in LEARNED_TEXT.items():
    cos = F.cosine_similarity(text[0], text[1], dim=0).item()
    print(f"  {name:26s} encoded -> {tuple(text.shape)}  cos(normal, abnormal) = {cos:+.4f}")

In [ ]:
# Score every prompt set. One visual forward pass per image, reused for all sets.
import numpy as np
from collections import defaultdict
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SIGMA = GAUSSIAN_SIGMA * MAP_RES / 518.0  # AnomalyCLIP specifies sigma at 518px.

def gaussian_blur(maps, sigma):
    """Separable Gaussian, matching AnomalyCLIP's inference smoothing."""
    if not sigma:
        return maps
    radius = max(1, int(4.0 * sigma + 0.5))
    radius = min(radius, maps.shape[-1] - 1, maps.shape[-2] - 1)
    grid = torch.arange(-radius, radius + 1, device=maps.device, dtype=maps.dtype)
    kernel = torch.exp(-(grid ** 2) / (2 * sigma * sigma))
    kernel = kernel / kernel.sum()
    x = maps[:, None]
    x = F.conv2d(F.pad(x, (radius, radius, 0, 0), mode="reflect"), kernel.view(1, 1, 1, -1))
    x = F.conv2d(F.pad(x, (0, 0, radius, radius), mode="reflect"), kernel.view(1, 1, -1, 1))
    return x[:, 0]

@torch.no_grad()
def score_category(samples, text_by_set):
    """Returns {set_name: (image_scores, pixel_maps)}, plus labels and masks."""
    loader = DataLoader(PromptTrainingDataset(samples, IMAGE_SIZE),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                        pin_memory=True)
    out = {name: {"score": [], "map": []} for name in text_by_set}
    labels, masks = [], []
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True)
        # One visual forward for the whole batch; every prompt set reuses it.
        global_features, patch_features = model.encode_visual(images)
        for name, text in text_by_set.items():
            image_logits, similarity_maps = model.predictions(
                global_features, patch_features,
                output_size=(MAP_RES, MAP_RES), text_features=text)
            # AnomalyCLIP's map: the abnormal channel, averaged over layers.
            anomaly = torch.stack([m[:, 1] for m in similarity_maps]).mean(0)
            anomaly = gaussian_blur(anomaly, SIGMA)
            out[name]["score"].append(image_logits.softmax(-1)[:, 1].float().cpu().numpy())
            out[name]["map"].append(anomaly.float().cpu().numpy().astype(np.float16))
        labels.append(batch["label"].numpy())
        # Max-pool keeps thin defects that a nearest resize would drop.
        small = F.adaptive_max_pool2d(batch["mask"].to(DEVICE), MAP_RES)
        masks.append((small[:, 0] > 0.5).cpu().numpy())
    return (
        {n: (np.concatenate(v["score"]), np.concatenate(v["map"])) for n, v in out.items()},
        np.concatenate(labels), np.concatenate(masks),
    )

RAW = defaultdict(dict)  # (dataset, protocol, category) -> payload
for (dataset, protocol), evaluation in sorted(EVAL_SETS.items()):
    names = GROUPS[(dataset, protocol)]
    categories = sorted({s.category for s in evaluation})
    print(f"\n=== {dataset}/{protocol}: {len(evaluation)} images, "
          f"{len(categories)} categories, {len(names) + 1} prompt sets ===", flush=True)
    for category in tqdm(categories, desc=f"{dataset}/{protocol}", unit="cat"):
        subset = [s for s in evaluation if s.category == category]
        text_by_set = {"frozen_winclip": frozen_text_for(category)}
        for name in names:
            text_by_set[name] = LEARNED_TEXT[name]
        scored, labels, masks = score_category(subset, text_by_set)
        RAW[(dataset, protocol)][category] = (scored, labels, masks)
print("\nscoring complete.")

In [ ]:
# Metrics: image AUROC / AP and pixel AUROC, per category then averaged.
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

rows = []
for (dataset, protocol), per_category in sorted(RAW.items()):
    for category, (scored, labels, masks) in sorted(per_category.items()):
        for name, (scores, maps) in scored.items():
            row = {"dataset": dataset, "protocol": protocol, "category": category,
                   "prompts": name, "n": len(labels)}
            row["image_auroc"] = (100 * roc_auc_score(labels, scores)
                                  if len(set(labels)) == 2 else float("nan"))
            row["image_ap"] = (100 * average_precision_score(labels, scores)
                               if len(set(labels)) == 2 else float("nan"))
            flat_mask, flat_map = masks.reshape(-1), maps.reshape(-1).astype(np.float32)
            row["pixel_auroc"] = (100 * roc_auc_score(flat_mask, flat_map)
                                  if flat_mask.any() and not flat_mask.all() else float("nan"))
            rows.append(row)

CATEGORY_TABLE = pd.DataFrame(rows)
METRICS = ["image_auroc", "image_ap", "pixel_auroc"]
SUMMARY = (CATEGORY_TABLE.groupby(["dataset", "protocol", "prompts"])[METRICS]
           .mean().round(2).reset_index())

pd.set_option("display.width", 160)
for (dataset, protocol), block in SUMMARY.groupby(["dataset", "protocol"]):
    print(f"\n=== {dataset} / {protocol}  (mean over categories, "
          f"{len(EVAL_SETS[(dataset, protocol)])} held-out images) ===")
    ordered = block.sort_values("pixel_auroc", ascending=False)
    print(ordered[["prompts"] + METRICS].to_string(index=False))

In [ ]:
# Verdict: (a) does training help, (b) does the run-to-run noise matter.
print("=" * 78)
print("(a) LEARNED vs FROZEN  -- does training the prompts help at all?")
print("=" * 78)
verdicts = []
for (dataset, protocol), block in SUMMARY.groupby(["dataset", "protocol"]):
    frozen = block[block.prompts == "frozen_winclip"]
    learned = block[block.prompts != "frozen_winclip"]
    if frozen.empty or learned.empty:
        continue
    for _, row in learned.iterrows():
        deltas = {m: row[m] - frozen.iloc[0][m] for m in METRICS}
        best = "learned" if deltas["pixel_auroc"] > 0 else "frozen"
        verdicts.append({"group": f"{dataset}/{protocol}", "prompts": row.prompts,
                         **{f"d_{m}": round(deltas[m], 2) for m in METRICS},
                         "pixel_winner": best})
        print(f"  {dataset}/{protocol:9s} {row.prompts:24s}  "
              + "  ".join(f"{m.replace('image_','img_').replace('pixel_','px_')}"
                          f" {deltas[m]:+6.2f}" for m in METRICS)
              + f"   -> {best}")

print()
print("=" * 78)
print("(b) SEED VARIANCE  -- two checkpoints, identical training images")
print("=" * 78)
found_repeat = False
for (dataset, protocol), block in SUMMARY.groupby(["dataset", "protocol"]):
    learned = block[block.prompts != "frozen_winclip"]
    if len(learned) < 2:
        continue
    found_repeat = True
    spread = {m: learned[m].max() - learned[m].min() for m in METRICS}
    print(f"  {dataset}/{protocol}: {len(learned)} repeats "
          f"({', '.join(learned.prompts)})")
    for m in METRICS:
        print(f"      {m:12s} range {learned[m].min():6.2f} - {learned[m].max():6.2f}"
              f"   spread {spread[m]:5.2f}")
    gap = spread["pixel_auroc"]
    print(f"      -> {'MATERIAL: report a variance bar and fix determinism' if gap > 1.0 else 'cosmetic: the parameter difference does not reach the metric'}")
if not found_repeat:
    print("  No group has two checkpoints. Attach a second run of the same"
          "\n  (dataset, protocol) -- e.g. your old and new balanced/mvtec -- to measure this.")

print()
print("=" * 78)
if verdicts:
    wins = sum(v["pixel_winner"] == "learned" for v in verdicts)
    print(f"SUMMARY: learned prompts beat the frozen ensemble on pixel AUROC in "
          f"{wins} of {len(verdicts)} comparisons.")
    if wins == 0:
        print("  -> Training is not earning its place. Fix training before anything else:")
        print("     lower training.learning_rate to 1e-4; try model.feature_map_indices [3];")
        print("     raise training.epochs (and selected_epoch) together.")
    elif wins == len(verdicts):
        print("  -> Training helps everywhere. Check (b) above before trusting the margin.")
    else:
        print("  -> Mixed. Compare per group above; the protocol/dataset matters.")
print("=" * 78)

In [ ]:
# Save the tables and one downloadable ZIP.
import shutil
from IPython.display import FileLink, display

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
CATEGORY_TABLE.to_csv(out / "per_category.csv", index=False)
SUMMARY.to_csv(out / "summary.csv", index=False)
pd.DataFrame(verdicts).to_csv(out / "verdict_learned_vs_frozen.csv", index=False)

(out / "run_metadata.json").write_text(json.dumps({
    "split_seed": SPLIT_SEED,
    "evaluation_fraction": EVALUATION_FRACTION,
    "map_res": MAP_RES,
    "gaussian_sigma_at_518": GAUSSIAN_SIGMA,
    "frozen_ensemble_sha256": frozen_ensemble_sha256(),
    "checkpoints": {n: {"path": str(p),
                        "dataset": pl["dataset"],
                        "protocol": pl["prompt_config"].get("split_protocol", "balanced"),
                        "cohort_sha256": pl["sample_manifest_sha256"]}
                    for n, (p, pl) in CHECKPOINTS.items()},
    "evaluation_sizes": {f"{d}/{p}": len(v) for (d, p), v in EVAL_SETS.items()},
}, indent=2, sort_keys=True) + "\n", encoding="utf-8")

archive = Path(shutil.make_archive("/kaggle/working/prompt_evaluation", "zip",
                                   root_dir=out.parent, base_dir=out.name))
print(f"wrote {archive} ({archive.stat().st_size / 1024:.0f} KiB)")
display(FileLink(str(archive)))

## Reading the output

`summary.csv` is the answer; `per_category.csv` is the breakdown behind it.

**Scoring path.** Both prompt modes reduce to the same thing -- a `[2, D]`
normalized text matrix, normal row first -- and go through `model.predictions`,
the identical function training used. The frozen ensemble is collapsed to two
prototypes by averaging unit embeddings, which is the attack pipeline's own
`mean_normalized_embedding_prototype` aggregation. Image score is the abnormal
softmax probability; the map is the abnormal channel averaged over the four
feature layers, Gaussian-smoothed as at inference.

**What is comparable.** Rows inside one (dataset, protocol) block share an
evaluation set and are directly comparable. Rows across protocols are not --
`balanced` and `full` hold out different images.

**Caveats.** Pixel metrics are computed at `MAP_RES`, not 518, so they are not
directly comparable with published AnomalyCLIP numbers; masks are max-pooled to
that resolution, which slightly dilates thin defects. Both choices apply equally
to every prompt set, so the comparison between them is unaffected.